<h1><font color="#113D68" size=6>Deep Learning con Python y Keras</font></h1>

<h1><font color="#113D68" size=5>Parte 5. Redes Neuronales Convolucionales</font></h1>

<h1><font color="#113D68" size=4>2. Reconocimiento de dígitos</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. MNIST dataset](#section1)
* [2. Cargar MNIST](#section2)
* [3. Modelo de línea de base con MLP](#section3)
* [4. CNN para MNIST](#section4)
* [5. CNN más profunda para MNIST](#section5)
* [6. LeNet5 para MNIST](#section6)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

En este proyecto, descubrirá cómo desarrollar un modelo de Deep Learning en la tarea de reconocimiento de dígitos manuscritos del MNIST. Después de completar esta clase sabrá:
* Cómo cargar MNIST y desarrollar un modelo de red neuronal.
* Cómo implementar y evaluar una CNN de línea base para MNIST.
* Cómo implementar un modelo de Deep Learning avanzado para MNIST.
* Como implementar un modelo del estado del arte.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. MNIST dataset</font>

MNIST toma imágenes de dígitos de una variedad de documentos escaneados, normalizados en tamaño y centrados.

Cada imagen es está dada en blanco y negro con $28 × 28$ píxeles (784 píxeles en total). Se usan 60,000 imágenes para entrenar un modelo y 10,000 imágenes para validarlo.

Es una tarea de reconocimiento de dígitos. Como tal, hay 10 dígitos (0 a 9) o 10 clases para predecir. 

En la página web de Rodrigo Benenson hay una lista de los resultados más avanzados y enlaces a los artículos relevantes sobre el MNIST y otros conjuntos de datos.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre el dataset [MNIST](http://yann.lecun.com/exdb/mnist/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Información de los resultados sobre MNIST de [Rodrigo Benenson](http://rodrigob.github.io/are_we_there_yet/build/classification_datasets_results.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Información de los resultados sobre [LeNet5](http://yann.lecun.com/exdb/lenet/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Kaggle: [_Digit Recognizer: Learn computer vision fundamentals with the famous MNIST data_](http://yann.lecun.com/exdb/mnist/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Artículo científico: Yann LeCun, Leon Bottou, Yoshua Bengio, and Patrick Haffner, [_Gradient-Based Learning Applied to Document Recognition_](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Cargar MNIST</font>

El conjunto de datos se descarga automáticamente la primera vez con `torchvision.datasets.MNIST(..., download=True)` y se guarda en la carpeta indicada (aquí `./data`, unos 15 MB).

Primero escribiremos un pequeño script para descargar y visualizar las primeras imágenes.

In [ ]:
import matplotlib.pyplot as plt
import torchvision

train = torchvision.datasets.MNIST('./data', train=True, download=True)

fig, ax = plt.subplots(4, 4, sharex=True, sharey=True)
for i in range(4):
    for j in range(4):
        ax[i][j].imshow(train.data[4*i+j], cmap="gray")
plt.show()

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. MLP de línea base</font>

Vamos a usar un MLP clásico como base para la comparación con modelos de redes neuronales convolucionales. 

Importamos las clases, funciones y el dataset MNIST.

In [ ]:
import torchvision

# load data
import torch
import torch.nn as nn
import torch.optim as optim

train = torchvision.datasets.MNIST('./data', train=True, download=True)
test = torchvision.datasets.MNIST('./data', train=False, download=True)

# aplanar 28x28 -> vector de 784 y pasar a float
X_train = train.data.reshape(-1, 784).float()
X_test = test.data.reshape(-1, 784).float()

Para un MLP clásico debemos reducir las imágenes a un vector de píxeles. En este caso, las imágenes de tamaño $28 × 28$ serán vectores de entrada de 784 píxeles. 

Realizamos esta transformación meidante la función `reshape()`. 

Los valores de los píxeles son números enteros, por lo que los convertimos a punto flotante para poder normalizarlos.

Los valores de los píxeles están en una escala de grises entre 0 y 255. Podemos normalizar los valores de los píxeles en el rango 0 y 1 dividiendo cada valor por el máximo valor, i.e., 255.

In [ ]:
# normalize inputs from 0-255 to 0-1
X_train = X_train / 255.0
X_test = X_test / 255.0

Finalmente, la variable de salida es un número entero de 0 a 9. Usaremos One-Hot Encoding para transformar el vector de enteros de clase en una matriz binaria con `torch.nn.functional.one_hot()` (equivalente al antiguo `to_categorical()` de Keras).

> En PyTorch esto es opcional: `nn.CrossEntropyLoss` también acepta directamente los índices de clase (0-9).

In [ ]:
# one hot encode outputs
y_train = torch.nn.functional.one_hot(train.targets, num_classes=10).float()
y_test = test.targets  # para test dejamos el índice de clase (se compara con argmax)

Vamos a definir nuestro modelo:
1. El número de entradas será el tamaño máximo de pixeles (784)
2. Tendrá una capa oculta con el mismo número de neuronas que entradas (784). 
3. Se utiliza una función de activación ReLU en la capa oculta. 
4. La salida de este modelo son _logits_, es decir, son números reales que se pueden transformar en valores similares a probabilidades usando una función softmax. No es necesario aplicar la función softmax de manera explícita, ya que la función de entropía cruzada lo hará por ti. 
5. La función de pérdida será `CrossEntropy`. 
6. Utilizaremos SGD para aprender los pesos con un _lr_ de 001. 

<center>
<figure>
  <img src="Datasets/baselineMLP.png" width="350" height="100" alt="Netron">
  <figcaption><blockquote>Diseño del MLP</blockquote></figcaption>
</figure>
</center>

In [ ]:
# define baseline model
class Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 784)
        self.act1 = nn.ReLU()
        self.layer2 = nn.Linear(784, 10)

    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.layer2(x)  # logits
        return x

model = Baseline()

optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()  # acepta one-hot (probabilidades) o índices
loader = torch.utils.data.DataLoader(list(zip(X_train, y_train)), shuffle=True, batch_size=100)

print(model)

Entrenamos y evaluamos el modelo. 
1. El modelo se ajusta a más de 10 épocas. 
2. Los datos de test se utilizan como conjunto de datos de validación.
3. Evaluamos en test e imprimimos las métricas.

In [ ]:
n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Validation
    model.eval()
    y_pred = model(X_test)
    acc = (torch.argmax(y_pred, 1) == y_test).float().mean()
    print("Epoch %d: model accuracy %.2f%%" % (epoch, acc*100))

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section4"></a>
# <font color="#004D7F" size=6>4. CNN para MNIST </font>

Ahora que hemos visto cómo cargar el conjunto de datos MNIST y entrenar un modelo simple de perceptrón multicapa en él, es hora de desarrollar una red neuronal convolucional más sofisticada o un modelo CNN. 

En PyTorch, se espera que las capas convolucionales trabajen con imágenes. Los tensores para imágenes deben contener los valores de los píxeles con las dimensiones (muestra, canal, altura, ancho), pero cuando cargas imágenes utilizando bibliotecas como PIL, los píxeles suelen estar organizados como un arreglo con dimensiones (altura, ancho, canal). La conversión al formato adecuado de tensor se puede hacer utilizando una transformación de la biblioteca torchvision.


En el caso de RGB, los canales de última dimensión serían 3 para los componentes rojo, verde y azul y sería como tener 3 entradas de imagen para cada imagen en color. En el caso de MNIST, donde los valores de los canales están en escala de grises, la dimensión de píxel se establece en 1.

Establecemos el shuffle y el batch_size a 100

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

# Load MNIST data
transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])  # (1, 28, 28) en [0, 1]
train = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test =  torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(train, shuffle=True, batch_size=100)
testloader = torch.utils.data.DataLoader(test, shuffle=False, batch_size=100)

A continuación, definimos nuestro modelo de red neuronal:

1. La primera capa oculta es una capa convolucional llamada `Conv2D`. 
    * Tiene 32 mapas de características, con un tamaño de $5 × 5$ y una función de activación ReLu.
    * El stride es 1, lo que significa que el kernel se desplaza un píxel a la vez a lo largo de la imagen de entrada.
    * El padding es 2, lo que significa que se agregarán dos ceros a cada lado de la imagen de entrada. Esto ayuda a mantener las dimensiones de la salida iguales a las de la entrada, ya que la convolución con un kernel de tamaño 5 reducirá las dimensiones en 4 píxeles.
2. Capa Pooling  `MaxPooling2D `. 
    * Tamaño de pacht de $2 × 2$.
    * Le indicaremos como `stride = 1`.
    * El resultado es un mapa de características de 27x27 por canal.
3. Capa de regularización `Dropout`. 
4. Capa `Flatten` para conversión de la matriz 2D en un vector (1D).
5. Capa `Dense` con 128 neuronas y la función de activación ReLU.
6. Capa de salida con 10 neuronas para las 10 clases y una función de activación **Softmax**.
7. La compilación con ADAM, pérdida logarítmica como función de coste y Accuracy como métrica.

<center>
<figure>
  <img src="Datasets/cnnTopology.png" width="750" height="100" alt="Netron">
  <figcaption><blockquote>Diseño de una CNN base</blockquote></figcaption>
</figure>
</center>

> **Nota (PyTorch):** la última capa devuelve **logits** (sin `Softmax`), porque `nn.CrossEntropyLoss` ya aplica `log_softmax` internamente. La “compilación” de Keras equivale aquí a elegir optimizador (`Adam`) y pérdida (`CrossEntropyLoss`).

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 32, kernel_size=(5, 5), stride=1, padding=2)  # 1x28x28 -> 32x28x28
        self.relu1 = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=(2, 2), stride=1)  # -> 32x27x27
        self.dropout = nn.Dropout(0.2)
        self.flat = nn.Flatten()
        self.fc = nn.Linear(32 * 27 * 27, 128)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = self.relu1(self.conv(x))
        x = self.pool(x)
        x = self.dropout(x)
        x = self.relu2(self.fc(self.flat(x)))
        x = self.output(x)
        return x

model = CNN()

optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

Entrenamos con 10 épocas.

In [ ]:
n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in trainloader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Validation
    model.eval()
    acc = 0
    count = 0
    for X_batch, y_batch in testloader:
        y_pred = model(X_batch)
        acc += (torch.argmax(y_pred, 1) == y_batch).float().sum()
        count += len(y_batch)
    acc = acc / count
    print("Época %d: Accuracy del modelo %.2f%%" % (epoch, acc*100))

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section5"></a>
# <font color="#004D7F" size=6>5. CNN más profunda para MNIST </font>

Esta vez definimos una arquitectura con más capas de convolucionales, Max-pooling y capas completamente conectadas.

1. Capa convolucional con 30 mapas de tamaño $5 × 5$.
2. Capa de Pooling con patch de $2 × 2$.
3. Capa convolucional con 15 mapas de tamaño $3 × 3$.
4. Capa de Pooling con patch de $2 × 2$.
5. Capa de Dropout del 20%.
6. Capa Flatten.
7. Capa completamente conectada con 128 neuronas y ReLu.
8. Capa completamente conectada con 50 neuronas y ReLu
9. Capa de salida cpm activación Softmax.
10. La compilación con ADAM, pérdida logarítmica como función de coste y Accuracy como métrica.

<center>
<figure>
  <img src="Datasets/cnnlarger.png" width="950" height="100" alt="Netron">
  <figcaption><blockquote>Diseño de una CNN más profunda</blockquote></figcaption>
</figure>
</center>

> **Nota (PyTorch):** la última capa devuelve **logits** (sin `Softmax`), porque `nn.CrossEntropyLoss` ya aplica `log_softmax` internamente. La “compilación” de Keras equivale aquí a elegir optimizador (`Adam`) y pérdida (`CrossEntropyLoss`).

In [ ]:
class LargerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 30, kernel_size=(5, 5), stride=1, padding=2)  # -> 30x28x28
        self.relu1 = nn.ReLU()
        self.pool1 =nn.MaxPool2d(kernel_size=(2, 2))  # -> 30x14x14
        self.conv2 = nn.Conv2d(30, 15, kernel_size=(3, 3), stride=1, padding=1)  # -> 15x14x14
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2))  # -> 15x7x7
        self.dropout = nn.Dropout(0.2)
        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(15 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 50)
        self.relu4 = nn.ReLU()
        self.output = nn.Linear(50, 10)

    def forward(self, x):
        x = self.relu1(self.conv1(x))

        x = self.pool1(x)

        x = self.relu2(self.conv2(x))

        x = self.pool2(x)

        x = self.dropout(x)

        x = self.flat(x)
        x = self.relu3(self.fc1(x))
        x = self.relu4(self.fc2(x))
        x = self.output(x)
        return x

model = LargerModel()

optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

Ahora vemos los resultados

In [ ]:
n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in trainloader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    acc = 0
    count = 0
    for X_batch, y_batch in testloader:
        y_pred = model(X_batch)
        acc += (torch.argmax(y_pred, 1) == y_batch).float().sum()
        count += len(y_batch)
    acc = acc / count
    print("Época %d: Accuracy del modelo %.2f%%" % (epoch, acc*100))

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section6"></a>
# <font color="#004D7F" size=6>6. LeNet5 para MNIST </font>

Este modelo fue desarrollado para resolver el problema de clasificación de MNIST. Tiene tres capas convolucionales y dos capas completamente conectadas, lo que suma cinco capas entrenables en el modelo, de ahí su nombre. En la época en que fue desarrollado, era común usar la función tangente hiperbólica como activación.

<center>
<figure>
  <img src="Datasets/lenet5.png" width="350" height="100" alt="Netron">
  <figcaption><blockquote>Arquitectura de la LeNet5</blockquote></figcaption>
</figure>
</center>

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=(5, 5), stride=1, padding=2)  # -> 6x28x28
        self.act1 = nn.Tanh()
        self.pool1 = nn.AvgPool2d(kernel_size=(2, 2), stride=2)  # -> 6x14x14

        self.conv2 = nn.Conv2d(6, 16, kernel_size=(5, 5), stride=1, padding=0)  # -> 16x10x10
        self.act2 = nn.Tanh()
        self.pool2 = nn.AvgPool2d(kernel_size=(2, 2), stride=2)  # -> 16x5x5

        self.conv3 = nn.Conv2d(16, 120, kernel_size=(5, 5), stride=1, padding=0)  # -> 120x1x1
        self.act3 = nn.Tanh()

        self.flat = nn.Flatten()
        self.fc1 = nn.Linear(120, 84)
        self.act4 = nn.Tanh()
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):

        x = self.act1(self.conv1(x))

        x = self.pool1(x)

        x = self.act2(self.conv2(x))

        x = self.pool2(x)

        x = self.act3(self.conv3(x))

        x = self.act4(self.fc1(self.flat(x)))

        x = self.fc2(x)
        return x

model = LeNet5()

optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

Ahora procedemos al entrenamiento.

In [ ]:
n_epochs = 10
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in trainloader:
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    acc = 0
    count = 0
    for X_batch, y_batch in testloader:
        y_pred = model(X_batch)
        acc += (torch.argmax(y_pred, 1) == y_batch).float().sum()
        count += len(y_batch)
    acc = acc / count
    print("Epoch %d: model accuracy %.2f%%" % (epoch, acc*100))


<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>